#### Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple
from scipy.stats import norm, invgamma, multivariate_normal
import pickle
import os

warnings.filterwarnings('ignore')
# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

import contextlib
import io

from kama_msr import KAMA
from kama_msr import MarkovSwitchingModel
from kama_msr import KAMA_MSR
from kmrf import KMRF
from KMRF_training_config import *

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  Pandas version: 2.2.1
  NumPy version: 1.26.4


## Model Data and Date Options

In [2]:
# Prepare data: etf_close_prices, commodity_close_prices, universe_close_prices
international_index_symbol_names = pd.read_csv('data/inputs/fmp_index_list.csv').set_index('symbol')['name']
international_index_symbol_names = international_index_symbol_names[~international_index_symbol_names.index.isin(['^GSPC', '^NDX'])].to_dict()
commodity_symbol_names = pd.read_csv('data/inputs/fmp_commodity_list.csv').set_index('symbol')['name'].to_dict()
etf_symbol_names = {
    # BOND ETFS
    'BIL': 'SPDR Bloomberg 1-3 Month T-Bill ETF',
    'SHY': 'iShares 1-3 Year Treasury Bond ETF',
    'IEF': 'iShares 7-10 Year Treasury Bond ETF',
    # International EQUITY ETFS
    'VXUS': 'Vanguard Total International Stock ETF',
    'VEA': 'Vanguard FTSE Developed Markets ETF',
    'VWO': 'Vanguard FTSE Emerging Markets ETF',
    'VGK': 'Vanguard FTSE Europe ETF',
    'VPL': 'Vanguard FTSE Pacific ETF',
    'FXI': 'iShares China Large-Cap ETF',
    'EWJ': 'iShares MSCI Japan ETF',
    'INDA': 'iShares MSCI India ETF',
    # MAJOR INDICES
    '^GSPC': 'S&P 500',
    '^IXIC': 'Nasdaq Composite',
    '^NDX': 'Nasdaq 100',
    '^RUT': 'Russell 2000',
    '^DJI': 'Dow Jones Industrial Average',
    '^RUI': 'Russell 1000',
    '^RUA': 'Russell 3000',
    
    # MAIN BROAD MARKET ETFS
    'SPY': 'SPDR S&P 500 ETF',
    'VOO': 'Vanguard S&P 500 ETF',
    'RSP': 'Invesco S&P 500 Equal Weight ETF',
    'IVV': 'iShares Core S&P 500 ETF',
    'QQQ': 'Invesco QQQ Trust',
    'QQQM': 'Invesco Nasdaq 100 ETF',
    'ONEQ': 'Fidelity Nasdaq Composite Index ETF',
    'IWM': 'iShares Russell 2000 ETF',
    'IWB': 'iShares Russell 1000 ETF',
    'IWV': 'iShares Russell 3000 ETF',
    'DIA': 'SPDR Dow Jones Industrial Average ETF',
    'VTI': 'Vanguard Total Stock Market ETF',
    
    # S&P 500 SECTOR ETFS (SELECT SECTOR SPDRS)
    'XLE': 'Energy Select Sector SPDR',
    'XLF': 'Financial Select Sector SPDR',
    'XLU': 'Utilities Select Sector SPDR',
    'XLI': 'Industrial Select Sector SPDR',
    'XLV': 'Health Care Select Sector SPDR',
    'XLK': 'Technology Select Sector SPDR',
    'XLB': 'Materials Select Sector SPDR',
    'XLY': 'Consumer Discretionary Select Sector SPDR',
    'XLP': 'Consumer Staples Select Sector SPDR',
    'XLRE': 'Real Estate Select Sector SPDR',
    'XLC': 'Communication Services Select Sector SPDR',
    
    # GROWTH ETFs
    'IVW': 'iShares S&P 500 Growth ETF',
    'VONG': 'Vanguard Russell 1000 Growth ETF',
    'IWF': 'iShares Russell 1000 Growth ETF',
    'IWO': 'iShares Russell 2000 Growth ETF',
    'VUG': 'Vanguard Growth ETF',
    'SPYG': 'SPDR Portfolio S&P 500 Growth ETF',
    
    # VALUE ETFs
    'IVE': 'iShares S&P 500 Value ETF',
    'VONV': 'Vanguard Russell 1000 Value ETF',
    'IWD': 'iShares Russell 1000 Value ETF',
    'IWN': 'iShares Russell 2000 Value ETF',
    'VTV': 'Vanguard Value ETF',
    'SPYV': 'SPDR Portfolio S&P 500 Value ETF',
    
    # SIZE ETFs
    'IWR': 'iShares Russell Mid-Cap ETF',
    'IWC': 'iShares Micro-Cap ETF',
    'IJH': 'iShares Core S&P Mid-Cap ETF',
    'IJR': 'iShares Core S&P Small-Cap ETF',
    'MDY': 'SPDR S&P MidCap 400 ETF',
    'SLY': 'SPDR S&P 600 Small Cap ETF',
    'VO': 'Vanguard Mid-Cap ETF',
    'VB': 'Vanguard Small-Cap ETF',
    'SCHA': 'Schwab U.S. Small-Cap ETF',
    'SCHM': 'Schwab U.S. Mid-Cap ETF',
    'VTWO': 'Vanguard Russell 2000 ETF',
    'VTHR': 'Vanguard Russell 3000 ETF',
    'THRK': 'iShares Russell 3000 ETF',
    'SPSM': 'SPDR Portfolio S&P 600 Small Cap ETF',
    'SMLF': 'iShares Small-Cap US Equity Factor ETF',
    
    # NASDAQ SPECIFIC
    'QTEC': 'First Trust Nasdaq-100 Technology Sector Index Fund',
    'QQEW': 'First Trust Nasdaq-100 Equal Weighted Index Fund',
    'QQQG': 'Pacer Nasdaq 100 Top 50 Cash Cows Dividend Growth ETF',
    'QQQV': 'Pacer Nasdaq 100 Top 50 Value ETF',
    
    # DIVIDEND/QUALITY
    'SCHD': 'Schwab U.S. Dividend Equity ETF',
    'VYM': 'Vanguard High Dividend Yield ETF',
    'DVY': 'iShares Select Dividend ETF',
    'QUAL': 'iShares MSCI USA Quality Factor ETF',
    'USMV': 'iShares MSCI USA Min Vol Factor ETF',
    
    # EQUAL WEIGHT
    'EWSC': 'Invesco S&P SmallCap 600 Equal Weight ETF',
    'EWMC': 'Invesco S&P MidCap 400 Equal Weight ETF',
}
universe_symbol_names = {
    'IVV': 'IVV - iShares Core S&P 500 ETF',
    'IJH': 'IJH - iShares Core S&P Mid-Cap ETF',
    'IWM': 'IWM - iShares Russell 2000 ETF',
    'EFA': 'EFA - iShares MSCI EAFE ETF',
    'EEM': 'EEM - iShares MSCI Emerging Markets ETF',
    'AGG': 'AGG - iShares Core U.S. Aggregate Bond ETF',
    'SPTL': 'SPTL - SPDR Portfolio Long Term Treasury ETF',
    'HYG': 'HYG - iShares iBoxx $ High Yield Corporate Bond ETF',
    'SPBO': 'SPBO - SPDR Portfolio Corporate Bond ETF',
    'IYR': 'IYR - iShares U.S. Real Estate ETF',
    'DBC': 'DBC - Invesco DB Commodity Index Tracking Fund',
    'GLD': 'GLD - SPDR Gold Shares',
}

# international_index_data = pd.read_csv('data/processed/index_data.csv', index_col=0, header=[0, 1], parse_dates=True)
commodity_data = pd.read_csv('data/processed/commodity_data.csv', index_col=0, header=[0, 1], parse_dates=True)
etf_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)
universe_data = pd.read_csv('data/processed/universe_etfs.csv', index_col=0, header=[0, 1], parse_dates=True)

commodity_data_close_cols = commodity_data.columns[commodity_data.columns.get_level_values(1) == 'close']
commodity_close_prices = commodity_data[commodity_data_close_cols].droplevel(1, axis=1).rename(columns=commodity_symbol_names)
commodity_close_prices.columns = [col.replace('/', ' ') for col in commodity_close_prices.columns]

etf_close_cols = etf_data.columns[etf_data.columns.get_level_values(1) == 'close']
etf_close_prices = etf_data[etf_close_cols].droplevel(1, axis=1).rename(columns=etf_symbol_names)

universe_close_cols = universe_data.columns[universe_data.columns.get_level_values(1) == 'close']
universe_close_prices = universe_data[universe_close_cols].droplevel(1, axis=1).rename(columns=universe_symbol_names)

# print('Commodities:', commodity_close_prices.columns.tolist())
# print('ETFs:', etf_close_prices.columns.tolist())
# print('Universe:', universe_close_prices.columns.tolist())

In [3]:
# Create asset_names_df
from KMRF_training_config import *
asset_names_df = pd.DataFrame({
    'universe': get_assets_by_class('universe') + ['']*7,
    'us_equity': get_assets_by_class('us_equity'),
    'commodity': get_assets_by_class('commodity') + ['']*5,
    'int_equity': get_assets_by_class('int_equity') + ['']*11,

})

# from pandas import option_context
# with option_context('display.max_colwidth', None):
#     display(asset_names_df)

In [4]:
rebal_dates = etf_close_prices['SPDR S&P 500 ETF'].to_frame().dropna().loc['2018-12-31':].index[::21].map(lambda dte: dte.strftime('%Y%m%d')).to_list()
rebal_dates[:5]

['20181231', '20190131', '20190304', '20190402', '20190502']

In [5]:
asset_names_df['us_equity'].tolist()

['SPDR S&P 500 ETF',
 'Invesco QQQ Trust',
 'iShares Russell 2000 ETF',
 'SPDR Dow Jones Industrial Average ETF',
 'Energy Select Sector SPDR',
 'Financial Select Sector SPDR',
 'Utilities Select Sector SPDR',
 'Industrial Select Sector SPDR',
 'Health Care Select Sector SPDR',
 'Technology Select Sector SPDR',
 'Materials Select Sector SPDR',
 'Consumer Discretionary Select Sector SPDR',
 'Consumer Staples Select Sector SPDR',
 'iShares S&P 500 Growth ETF',
 'iShares S&P 500 Value ETF',
 'iShares Russell 2000 Growth ETF',
 'iShares Russell 2000 Value ETF',
 'iShares Russell Mid-Cap ETF',
 'iShares Micro-Cap ETF']

## Load KAMA+MSR and KMRF Models

In [46]:
CONFIG = KMRF_Training_Config(
    asset_name= asset_names_df['us_equity'].tolist()[0], # Update this to desired asset
    classification_type='original',
    use_data_type='master',
    end_date=rebal_dates[0], # Update this to desired rebal data
    feature_window_size=1,
    feature_asset_classes=[],
    cross_asset_specific=[], 
    use_boruta_selection=True,
    use_consensus_selection=False
)
kama_msr_file_path = f"saved_models/KAMA_MSR/{CONFIG.get_asset_class()}/{CONFIG.get_date_ranges()['end_date']}/{CONFIG.get_asset_name()}_KAMA-MSR_4-regimes.pkl"
kmrf_file_path = f"saved_models/KMRF_new/original/{CONFIG.get_asset_class()}/{CONFIG.get_asset_name().replace(' ', '_')}_KMRF_model.pkl"
with open(kama_msr_file_path, 'rb') as f:
    kama_msr = pickle.load(f)
with contextlib.redirect_stdout(io.StringIO()):
    kmrf = KMRF.load_model(kmrf_file_path)

## Build HMM Posterior Regime Probailities

In [208]:
def get_forward_regime_probs(kama_msr: KAMA_MSR, kmrf: KMRF, alpha: float = 0.75): # confidence weight on likelihood vs prior)    
    with contextlib.redirect_stdout(io.StringIO()):
        ########## Fit exponential certainty decay to regime autocorrelations ##########
        regime_autocorrs = {}
        for k in range(1, 21):
            regime_autocorrs[k] = kama_msr.regime_labels.autocorr(lag=k)
            
        regime_autocorrs = pd.Series(regime_autocorrs)

        from scipy.optimize import curve_fit

        def exp_decay(delta, a, lambda_decay):
            return a * np.exp(-lambda_decay * delta)
        popt, pcov = curve_fit(exp_decay, regime_autocorrs.index, regime_autocorrs.values, p0=(0.5, 0.05))
        a_fit, lambda_decay_fit = popt
        
        ################################################################################
        
        transition_results = kama_msr.regime_transition_analysis(in_depth=False) 
        P = transition_results[1]['transition_matrix'] + 1 # Add 1 for Laplace Smoothing
        P = P.apply(lambda row: row / row.sum(), axis=1)
        eigvals, eigvecs = np.linalg.eig(P.T)
        steady_state = eigvecs[:, np.isclose(eigvals, 1)]
        steady_state = steady_state[:, 0].real
        steady_state /= steady_state.sum()
        
        kmrf_likelihood_t1 = kmrf.predict(test_or_val='val').iloc[0].reset_index(drop=True).rename('')

        prior_t1 = P.loc[kama_msr.regime_labels[-1]].reset_index(drop=True).rename('') # P(regime_t+1 | regime_t)
        
        unnormalized = prior_t1.pow(1-alpha).multiply(kmrf_likelihood_t1.pow(alpha))
        posterior_t1 = unnormalized / unnormalized.sum()
    
        forward_probs = np.zeros((21, 4))
        uncertainty_factor = 1 - a_fit*np.exp(-lambda_decay_fit*1)
        forward_probs[0] = (1 - uncertainty_factor) * posterior_t1 + uncertainty_factor * steady_state
        for i in range(1, 21):
            delta = i+1
            uncertainty_factor = 1 - a_fit*np.exp(-lambda_decay_fit*delta)
            probs_raw_delta = forward_probs[i-1] @ P.values
            forward_probs[i] = (1 - uncertainty_factor) * probs_raw_delta + uncertainty_factor * steady_state
    
    day_t = kama_msr.regime_labels.index[-1].strftime('%Y-%m-%d')
    new_index = kmrf.raw_ohlc.loc[day_t:][1:22].index
    df = pd.DataFrame(forward_probs, index=new_index, columns=range(0, 4)).rename_axis('regime', axis=1)
    df['uncertainty_factor'] = [1- a_fit*np.exp(-lambda_decay_fit*delta) for delta in range(0, 21)]
    return df

In [207]:
get_forward_regime_probs(kama_msr, kmrf)

regime,0,1,2,3,uncertainty_factor
date,,,,,
2019-01-02,0.050342,0.819708,0.009151,0.120799,0.037492
2019-01-03,0.114544,0.758963,0.008101,0.118392,0.057011
2019-01-04,0.184238,0.694024,0.007502,0.114235,0.076133
2019-01-07,0.256212,0.627799,0.007209,0.108780,0.094868
2019-01-08,0.327488,0.562910,0.007112,0.102490,0.113223
2019-01-09,0.395520,0.501543,0.007134,0.095803,0.131205
2019-01-10,0.458326,0.445349,0.007218,0.089108,0.148823
2019-01-11,0.514543,0.395411,0.007329,0.082717,0.166084
2019-01-14,0.563423,0.352272,0.007444,0.076861,0.182995


In [147]:
df = kama_msr.regime_labels.to_frame('regime').join(kama_msr.returns.rename('return')).dropna()
return_characterics_by_regime = df.groupby('regime')['return'].agg(['count', 'mean', 'std', 'skew', lambda ser: 3+ser.kurtosis()])\
    .rename(columns={'<lambda_0>': 'kurtosis'})
mean_by_regime = return_characterics_by_regime['mean']
var_by_regime = return_characterics_by_regime['std'] ** 2
return_characterics_by_regime

,count,mean,std,skew,kurtosis
regime,,,,,
0,4548,0.000473,0.008434,-0.345051,4.535855
1,1128,0.000494,0.012768,0.059126,3.074286
2,44,-0.008867,0.032651,0.161388,2.611367
3,303,-0.002151,0.031149,0.266104,4.332219
